# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library and dataset schema defined by Croissant.

### Dataset Source
The dataset is defined by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the Croissant dataset object
dataset = mlc.Dataset(croissant_url)

# Print out the name and description of the dataset
print("Dataset Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id` values as defined in the schema.

In [ ]:
# List all available RecordSets and their properties
print("Record sets found in the dataset metadata:")
record_sets = list(dataset.metadata.record_sets())
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, Name: {field.name}, Data Type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from selected record set(s) into Pandas DataFrames using their `@id` values for programmatic reference.

In [ ]:
# For this dataset, we'll programmatically extract all tabular record sets
dataframes = {}

# Identify tabular record sets and extract with mlcroissant
for rs in dataset.metadata.record_sets():
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"RecordSet @id: {rs.id} loaded with shape {df.shape} and fields:")
        print(df.columns.tolist())
    except Exception as e:
        print(f"Could not load RecordSet {rs.id} due to: {e}")
        continue
print("\nAll available RecordSet @id keys:")
print(list(dataframes.keys()))

# Preview the first (main survey responses) RecordSet, by @id
main_rs_id = list(dataframes.keys())[0] if dataframes else None
if main_rs_id:
    print(f"\nSample from main record set (@id={main_rs_id}):")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate some typical data analysis operations.

We'll select one numeric field and one categorical field by `@id` (see cell above for options) and show filtering, normalization, and grouping.

In [ ]:
# For demonstration, we select 'phq9_total_score' as numeric, and 'gender' as group
# The actual column names may differ; update based on printed DataFrame above
if main_rs_id is not None:
    df = dataframes[main_rs_id].copy()

    # Guess likely column ids -- adjust if columns differ
    possible_numeric_ids = [col for col in df.columns if ('phq' in col or 'score' in col or 'gad' in col or 'psq' in col) and pd.api.types.is_numeric_dtype(df[col])]
    if possible_numeric_ids:
        numeric_field_id = possible_numeric_ids[0]
    else:
        numeric_field_id = df.select_dtypes('number').columns[0]

    # Try to find a demographic field (e.g., gender, village) for grouping
    possible_group_fields = [c for c in df.columns if 'gender' in c.lower() or 'village' in c.lower() or 'education' in c.lower()]
    group_field_id = possible_group_fields[0] if possible_group_fields else df.columns[0]

    # Remove NA
    df = df.dropna(subset=[numeric_field_id])

    threshold = df[numeric_field_id].mean()  # Use mean as threshold for demo
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtering records in main RecordSet '@id': {main_rs_id} where '{numeric_field_id}' > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical, take means
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of a key numeric score (e.g., PHQ-9 or GAD-7 total score) and differences by demographic groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None:
    # Histogram of the selected numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group
    if group_field_id in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
- This notebook demonstrated loading FAIR²-compliant survey data using `mlcroissant`, exploring the available record sets, and conducting basic filtering, grouping, and visualizations.
- Data fields were referenced by their Croissant `@id` values for programmatic reproducibility.
- Analysts can extend this notebook by further investigating associations, handling missing values, or using more advanced statistics and machine learning tools!